## Libraries import

In [22]:
import os
from PIL import Image
from torch.utils.data import Dataset, Subset, DataLoader
from torchvision import transforms
import torchvision.models as models
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import copy

## Dataset Class

In [23]:
class CelebADataset(Dataset):
    def __init__(self, label_file, image_dir, partition_file, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.samples = []
        self.partition_map = {}
        self.train_indices = []
        self.val_indices = []
        self.test_indices = []

        with open(partition_file, 'r') as f:
            for line in f:
                img_name, partition = line.strip().split()
                base_name = os.path.splitext(img_name)[0]
                self.partition_map[base_name] = int(partition)

        with open(label_file, 'r') as f:
            for line in f:
                img_name, label = line.strip().split()
                self.samples.append((img_name, int(label)))

        for idx, (img_name, _) in enumerate(self.samples):
            base_name = os.path.splitext(img_name)[0]
            split = self.partition_map[base_name]
            if split == 0:
                self.train_indices.append(idx)
            elif split == 1:
                self.val_indices.append(idx)
            elif split == 2:
                self.test_indices.append(idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, label = self.samples[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

## Data Loading & Transform

In [24]:
base_path = '../AdvCelebA'
label_file = os.path.join(base_path, 'attack_CelebA.txt')
image_dir = os.path.join(base_path, 'images')
partition_file = os.path.join(base_path, 'list_eval_partition_no_overlap.txt')

transform = transforms.ToTensor()

dataset = CelebADataset(label_file, image_dir, partition_file, transform=transform)

train_dataset = Subset(dataset, dataset.train_indices)
val_dataset = Subset(dataset, dataset.val_indices)
test_dataset = Subset(dataset, dataset.test_indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(val_dataset, batch_size=32)

## Model

In [25]:
model = models.resnet18(pretrained=True)

# Fine-tune entire model
for param in model.parameters():
    param.requires_grad = True

# Replace the classification head for binary output
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 1)  # For BCEWithLogitsLoss
)

## Training Loop

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 30
patience = 2
best_val_loss = float('inf')
early_stop_counter = 0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} - Training"):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)
    val_loss /= len(val_loader.dataset)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        print(f"🔁 No improvement in validation loss for {early_stop_counter} epochs.")

        if early_stop_counter >= patience:
            print(f"⏹️ Early stopping triggered at epoch {epoch+1}")
            break

model.load_state_dict(best_model_wts)

Using device: mps


Epoch 1 - Training: 100%|██████████| 5072/5072 [10:01<00:00,  8.43it/s]


Epoch 1, Train Loss: 0.3542, Val Loss: 0.3356


Epoch 2 - Training: 100%|██████████| 5072/5072 [10:23<00:00,  8.14it/s]


Epoch 2, Train Loss: 0.3251, Val Loss: 0.3996
🔁 No improvement in validation loss for 1 epochs.


Epoch 3 - Training:  27%|██▋       | 1374/5072 [02:34<06:55,  8.90it/s]


KeyboardInterrupt: 

In [27]:
model.load_state_dict(best_model_wts)


<All keys matched successfully>

In [ ]:
torch.save(model.state_dict(), '../models_bin/resnet18_baseline.pth')

## Validation Accuracy

In [28]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        preds = torch.sigmoid(outputs).cpu().numpy() > 0.5
        all_preds.extend(preds.flatten())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {acc:.4f}")

Validation Accuracy: 0.8610
